<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/kinematics/07_differential_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K7 差動二輪ロボットの運動学と経路追従(notebook)

本文:[`07-differential-drive.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/kinematics/07-differential-drive.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

W = 0.3     # 左右の車輪の間隔(トレッド)[m]
DT = 0.02   # 時間のきざみ [s]

def wrap(th):
    return np.arctan2(np.sin(th), np.cos(th))               # 角度を [-pi, pi] に直す

## 1. 車輪の速さから、並進速度と角速度へ

本文の式 (1) と、その逆を書く。

**手計算**:本文の確認課題1・2を解く。

- 手計算:(ここに書く)

In [ ]:
def wheels_to_twist(v_l, v_r, w=W):
    raise NotImplementedError("TODO: v = (v_r + v_l) / 2、omega = (v_r - v_l) / w を返す")

def twist_to_wheels(v, omega, w=W):
    raise NotImplementedError("TODO: v_l = v - omega * w / 2、v_r = v + omega * w / 2 を返す")

In [ ]:
check("確認課題1:v = 0.5、omega = 1.0", wheels_to_twist(0.35, 0.65), [0.5, 1.0])
check("逆に直すと元に戻る", twist_to_wheels(*wheels_to_twist(0.35, 0.65)), [0.35, 0.65])
check("その場で回る:v = 0", wheels_to_twist(-0.2, 0.2)[0], 0.0)

## 2. オイラー法で動かす

本文の式 (3) で、状態 `pose = [x, y, theta]` を1ステップ進める関数を書く。角度は `wrap` で $[-\pi, \pi]$ に直す。

In [ ]:
def step(pose, v_l, v_r, dt=DT):
    x, y, th = pose
    v, omega = wheels_to_twist(v_l, v_r)
    raise NotImplementedError("TODO: 式 (3) で x, y, th を進め、th は wrap で直して np.array([x, y, th]) を返す")

In [ ]:
pose = np.zeros(3)
for _ in range(100):
    pose = step(pose, 0.5, 0.5)
check("左右同じ速さで 2 秒:x = 1 m まっすぐ", pose, [1.0, 0.0, 0.0])

v_l, v_r = twist_to_wheels(0.5, 1.0)                        # 半径 0.5 m の円
n_lap = int(round(2 * np.pi / 1.0 / DT))                    # 確認課題2:1周の時間 / DT
pose = np.zeros(3); traj = [pose]
for _ in range(n_lap):
    pose = step(pose, v_l, v_r); traj.append(pose)
traj = np.array(traj)
print(f"1周後の位置 {traj[-1].round(4)}(始点に戻るはず)")
check("1周後、始点の近くに戻る(オイラー法の誤差で少しずれる)", np.linalg.norm(traj[-1, :2]) < 0.02, True)
check("円の中心 (0, 0.5) からの距離は、ほぼ 0.5", np.abs(np.linalg.norm(traj[:, :2] - [0, 0.5], axis=1) - 0.5).max() < 0.02, True)
check("角度はいつも [-pi, pi]", np.abs(traj[:, 2]).max() <= np.pi, True)

plt.figure(figsize=(4, 4)); plt.plot(traj[:, 0], traj[:, 1]); plt.axis("equal"); plt.title("circle, radius 0.5 m"); plt.grid(); plt.show()

### 時間のきざみと誤差

**予想**:時間のきざみ `dt` を 0.2 秒、0.02 秒、0.002 秒にすると、10 周したあとの位置のずれはどうなるか。

- 予想:(ここに書く)

In [ ]:
for dt in [0.2, 0.02, 0.002]:
    pose = np.zeros(3)
    for _ in range(int(round(10 * 2 * np.pi / dt))):
        pose = step(pose, v_l, v_r, dt=dt)
    print(f"dt = {dt:5}:10 周後の始点からのずれ {np.linalg.norm(pose[:2]):.4f} m")

## 3. デッドレコニングの誤差

本当の車輪の速さには、指令の値に小さな雑音(標準偏差 0.02 m/s)が入るとする。指令の値だけから計算した位置(推定)と、本当の位置のずれを、40 秒間記録する(本文 §3)。seed を変えて 20 回くり返し、ずれの平均を見る。

**予想**:ずれは、時間に比例して増えるか、もっと速く増えるか。

- 予想:(ここに書く)

In [ ]:
def dead_reckoning(seed, T=2000, noise=0.02):
    rng = np.random.default_rng(seed)
    true_pose, est_pose = np.zeros(3), np.zeros(3)
    drift, heading_err = [], []
    for _ in range(T):
        v_l, v_r = 0.4, 0.5                                 # ゆるい左カーブ
        n = rng.normal(0, noise, 2)
        true_pose = step(true_pose, v_l + n[0], v_r + n[1])
        est_pose = step(est_pose, v_l, v_r)                 # 指令の値だけから推定する
        drift.append(np.linalg.norm(true_pose[:2] - est_pose[:2]))
        heading_err.append(abs(wrap(true_pose[2] - est_pose[2])))
    return np.array(drift), np.array(heading_err)

runs = [dead_reckoning(s) for s in range(20)]
drift = np.mean([d for d, _ in runs], axis=0); head = np.mean([h for _, h in runs], axis=0)
t = np.arange(len(drift)) * DT
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(t, drift); ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("position error [m] (20-run mean)")
ax[1].plot(t, head); ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("heading error [rad]")
plt.tight_layout(); plt.show()
for sec in [10, 20, 40]:
    print(f"{sec:2d} 秒後:位置のずれの平均 {drift[int(sec / DT) - 1]:.3f} m、向きのずれの平均 {head[int(sec / DT) - 1]:.3f} rad")

## 4. pure pursuit

本文 §4 を実装する。前方注視点の選び方(経路の上で、最も近い点から先へ進み、距離が $L_d$ 以上になる最初の点)は用意してある。

**手計算**:本文の確認課題4を解く。

- 手計算:(ここに書く)

**実装**:前方注視点 `(px, py)` をロボットから見た座標に直し、式 (4) の曲率から $\omega$ を計算する。

In [ ]:
def to_robot_frame(pose, point):
    x, y, th = pose
    raise NotImplementedError("TODO: 本文 §4 の回転で (x_L, y_L) を返す")

def pure_pursuit(pose, path, lookahead=0.5, v=0.4):
    d = np.linalg.norm(path - pose[:2], axis=1)
    i = int(np.argmin(d))
    while i < len(path) - 1 and d[i] < lookahead:
        i += 1
    x_L, y_L = to_robot_frame(pose, path[i])
    L_d = max(np.hypot(x_L, y_L), 1e-6)                      # 実際の距離(0 で割らないように)
    omega = v * 2 * y_L / L_d**2                             # 式 (4) と omega = v * kappa
    return twist_to_wheels(v, omega)

In [ ]:
check("確認課題4:(x_L, y_L) = (0.4, 0.3)", to_robot_frame([0, 0, np.pi / 2], [-0.3, 0.4]), [0.4, 0.3])
check("ロボットの真正面の点は (距離, 0)", to_robot_frame([1.0, 2.0, 0.7], [1.0 + np.cos(0.7), 2.0 + np.sin(0.7)]), [1.0, 0.0])

ts_ = np.linspace(0, 2 * np.pi, 400)
circle_path = np.column_stack([2 * np.cos(ts_), 2 * np.sin(ts_)])  # 半径 2 m の円の経路

def follow(path, lookahead, start=(2.2, -0.4, np.pi / 2), T=3000, cross_track=None):
    pose = np.array(start, dtype=float); traj, err = [], []
    for _ in range(T):
        pose = step(pose, *pure_pursuit(pose, path, lookahead))
        traj.append(pose); err.append(cross_track(pose) if cross_track else np.min(np.linalg.norm(path - pose[:2], axis=1)))
    return np.array(traj), np.array(err)

circle_err = lambda pose: abs(np.linalg.norm(pose[:2]) - 2.0)  # 円の経路からの横のずれ
traj, err = follow(circle_path, 0.5, cross_track=circle_err)
print(f"横のずれ:最初の1秒の平均 {err[:50].mean():.3f} m → 最後の10秒の平均 {err[-500:].mean():.4f} m")
check("経路から外れた所から始めても、経路に近づく", err[-500:].mean() < err[:50].mean(), True)
plt.figure(figsize=(4, 4)); plt.plot(*circle_path.T, "r--", label="path"); plt.plot(traj[:, 0], traj[:, 1], "b", label="robot")
plt.axis("equal"); plt.legend(); plt.title("pure pursuit, L_d = 0.5"); plt.show()

## 5. 条件を変える:前方注視距離

1. 円の経路で、 $L_d$ を 0.2、0.5、1.5 m に変えると、最後の 10 秒の横のずれはどうなるか
2. 四角形の経路(1辺 3 m)で、 $L_d$ を 0.3 m と 1.2 m にすると、角でどうなるか(本文 §4 の表)

In [ ]:
for ld in [0.2, 0.5, 1.5]:
    _, e = follow(circle_path, ld, cross_track=circle_err)
    print(f"1. L_d = {ld}:最後の10秒の横のずれの平均 {e[-500:].mean():.4f} m")

side = np.linspace(0, 3, 150)
square = np.vstack([np.column_stack([side, np.zeros(150)]), np.column_stack([np.full(150, 3), side]),
                    np.column_stack([side[::-1], np.full(150, 3)]), np.column_stack([np.zeros(150), side[::-1]])])
plt.figure(figsize=(5, 5)); plt.plot(*square.T, "k--", label="path")
for ld, c in [(0.3, "b"), (1.2, "r")]:
    tr, e = follow(square, ld, start=(0.0, -0.2, 0.0), T=1500)
    plt.plot(tr[:, 0], tr[:, 1], c, label=f"L_d = {ld}")
    print(f"2. L_d = {ld}:経路からのずれの最大 {e[100:].max():.3f} m")
plt.axis("equal"); plt.legend(); plt.title("square path"); plt.show()

## 6. 研究で使うときは

`cpai_lab` には移動ロボットの関数は入っていない(本文 §6)。ROS 2 で TurtleBot3 を動かすときは、この notebook と次のように対応する。

| この notebook | ROS 2 |
|---|---|
| `twist_to_wheels` に渡す `(v, omega)` | `cmd_vel` の `linear.x` と `angular.z` |
| `step` で計算した `est_pose`(指令の値からの推定) | `odom` の位置と向き(車輪の回転から計算。ずれていく) |
| 本当の位置 `true_pose` | シミュレータの中の正しい位置(実機では分からない。LiDAR などで推定する) |

## 7. 説明(自分の言葉で書く)

1. 差動二輪ロボットは、どうやって向きを変えるか。真横に動けないのはなぜか
   - (ここに書く)
2. デッドレコニングの位置のずれが、時間とともに増えていくのはなぜか。向きのずれは、位置のずれにどう影響するか
   - (ここに書く)
3. pure pursuit の前方注視距離を小さくした場合と大きくした場合の違いは何か
   - (ここに書く)